# Behind the scenes: Day Two
### Data download and preparation

1. Beck et al. ALREADY DONE IN DAY1_BTS!
2. GBIF species occurrence records (Philippine Eagle, Flying Fox) + a broader mammal community dataset
3. Sentinel-2 satellite imagery for Negros island
4. Hansen Global Forest Change data for Negros

In [ ]:
from pathlib import Path
import re
import requests
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
import rioxarray
from rasterio.windows import from_bounds
from rasterio.io import MemoryFile
from rasterio.merge import merge as rio_merge
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pyproj import Transformer
import pystac_client
import planetary_computer

from workshop_utils import RAW_DIR, PROCESSED_DIR
print(f'RAW_DIR: {RAW_DIR}')

---
## 1. Beck et al. (2023) — Köppen-Geiger maps and 1 km climate data

> Beck, H. E., et al. (2023). High-resolution (1 km) Köppen-Geiger maps

Already done!

---
## 2. GBIF species occurrence records

Two species, two purposes:
- **Philippine Eagle** (*Pithecophaga jefferyi*, GBIF taxonKey 2480381) — Day 2 biodiversity showcase.
  Even with few records, the map tells a powerful conservation story.
- **Philippine Flying Fox** (*Pteropus vampyrus*, GBIF taxonKey 5218644) — Day 3 SDM target.
  Widespread across the archipelago, with enough records for a meaningful model.

⚠️ Verify taxonKeys via `GET /v1/species/match?name=...` before use — GBIF backbone taxonomy
keys can go stale (the previous keys for both species silently returned 0 PH records).

No API key required. Pagination via `offset`.

In [ ]:
def download_gbif(taxon_key, country='PH', out_path=None):
    records, offset, limit = [], 0, 300
    base = 'https://api.gbif.org/v1/occurrence/search'
    while True:
        r = requests.get(base, params={
            'taxonKey': taxon_key, 'country': country,
            'hasCoordinate': 'true', 'hasGeospatialIssue': 'false',
            'limit': limit, 'offset': offset,
        })
        r.raise_for_status()
        data = r.json()
        records.extend(data['results'])
        print(f"  {len(records)}/{data['count']}", end='\r')
        if data['endOfRecords']:
            break
        offset += limit
    print()

    df = pd.DataFrame([
        {
            'species':  rec.get('species', ''),
            'lon':      rec.get('decimalLongitude'),
            'lat':      rec.get('decimalLatitude'),
            'year':     rec.get('year'),
            'basis':    rec.get('basisOfRecord', ''),
            'gbifID':   rec.get('gbifID'),
        }
        for rec in records if rec.get('decimalLongitude') is not None
    ])

    if out_path is not None:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_path, index=False)
        print(f'  saved {len(df)} records -> {out_path}')
    return df

In [ ]:
eagle = download_gbif(
    taxon_key=2480381,  # Pithecophaga jefferyi (verified via /v1/species/match; old key 2480965 was stale, returned 0 PH records)
    out_path=RAW_DIR / 'GBIF' / 'philippine_eagle.csv',
)
eagle.head()

In [ ]:
fox = download_gbif(
    taxon_key=5218644,  # Pteropus vampyrus (old key 2432859 was stale and returned 0 PH records)
    out_path=RAW_DIR / 'GBIF' / 'flying_fox.csv',
)
print(fox['year'].value_counts().sort_index().tail(15))
fox.head()

In [ ]:
# --- Clean + copy to PROCESSED_DIR/day_2 for student use ---
day2_dir = PROCESSED_DIR / "day_2"
day2_dir.mkdir(parents=True, exist_ok=True)

for label, fname in [("philippine_eagle", "philippine_eagle.csv"), ("flying_fox", "flying_fox.csv")]:
    df = pd.read_csv(RAW_DIR / "GBIF" / fname)
    df = (
        df.dropna(subset=["lon", "lat"])
          .drop_duplicates(subset=["gbifID"])
          .query("114 <= lon <= 127 and 4 <= lat <= 22")  # sanity-check: inside PH bounds
          .sort_values("year")
    )
    df.to_csv(day2_dir / f"gbif_{label}.csv", index=False)
    print(f"saved gbif_{label}.csv ({len(df)} rows)")

### 2b. GBIF mammal community dataset — urbanisation & terrestrial mammals in the Philippines

A much richer complement to the two single-species pulls above: a compiled GBIF occurrence
download of **all present-day terrestrial mammal records for the Philippines**
(class Mammalia, `COUNTRY=PH`, `OCCURRENCE_STATUS=present`), assembled from iNaturalist,
museum specimens, barcoding projects, etc. Backs the paper *"Making it in the city: Public
data reveal patterns and drivers of terrestrial mammal occurrence across urban Philippines."*

- 67,250 raw records → 289 species after cleaning
- Includes IUCN Red List category, island/province/municipality, and ~5,500 Negros-tagged
  records — good direct overlap with our Sentinel-2 / Hansen Negros data for Part IV
  (urban expansion vs. mammal occurrence)
- GBIF download DOI: [10.15468/dl.6ju8sc](https://doi.org/10.15468/dl.6ju8sc) — **cite this
  DOI (see `citations.txt` in the download) whenever the dataset is used**, since it is a
  compilation of many constituent datasets.

In [ ]:
import zipfile

GBIF_DOWNLOAD_KEY = "0003657-260226173443078"
GBIF_DOI_URL = f"https://api.gbif.org/v1/occurrence/download/request/{GBIF_DOWNLOAD_KEY}.zip"

mammals_dir = RAW_DIR / "GBIF" / "mammals_philippines"
mammals_dir.mkdir(parents=True, exist_ok=True)
zip_path = mammals_dir / f"{GBIF_DOWNLOAD_KEY}.zip"

if not zip_path.exists():
    print("downloading GBIF mammal occurrence download...")
    r = requests.get(GBIF_DOI_URL, timeout=120)
    r.raise_for_status()
    zip_path.write_bytes(r.content)
    print(f"saved {zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("skip download, zip exists")

with zipfile.ZipFile(zip_path) as z:
    for member in ["occurrence.txt", "citations.txt", "rights.txt"]:
        if not (mammals_dir / member).exists():
            z.extract(member, mammals_dir)
print("extracted occurrence.txt, citations.txt, rights.txt")

In [ ]:
# --- Process: trim to workshop-relevant columns, clean, save to PROCESSED_DIR/day_2 ---
KEEP_COLS = [
    "gbifID", "species", "genus", "family", "order", "class",
    "decimalLongitude", "decimalLatitude", "coordinateUncertaintyInMeters",
    "year", "month", "day", "basisOfRecord", "individualCount",
    "island", "islandGroup", "stateProvince", "county", "municipality", "locality",
    "iucnRedListCategory", "establishmentMeans", "institutionCode", "datasetKey",
]

mammals = pd.read_csv(mammals_dir / "occurrence.txt", sep="\t", usecols=KEEP_COLS, low_memory=False)
before = len(mammals)
mammals = mammals.dropna(subset=["decimalLongitude", "decimalLatitude", "species"])
mammals = mammals[mammals["class"] == "Mammalia"]
print(f"{before} -> {len(mammals)} rows after cleaning  ({mammals['species'].nunique()} species)")
print(mammals["iucnRedListCategory"].value_counts(dropna=False))

negros_mask = (mammals["county"].astype(str).str.contains("NEGROS", case=False, na=False) |
               mammals["stateProvince"].astype(str).str.contains("Negros", case=False, na=False))
print(f"Negros-tagged records: {negros_mask.sum()}")

out_csv = day2_dir / "gbif_mammals_philippines.csv"
mammals.to_csv(out_csv, index=False)
print(f"saved {out_csv}  ({out_csv.stat().st_size / 1e6:.1f} MB)")

---
## 3. Sentinel-2 satellite imagery — Negros island

Two low-cloud composites (historical ~2016-2018, recent ~2023-2024) over all of Negros.
Data from the **Element84 Earth Search** STAC catalog — no credentials needed.

⚠️ Negros (~130 km × 210 km) is bigger than a single Sentinel-2 tile (~110 km × 110 km),
so it straddles **six MGRS tiles** (`51PVL/VM/VN/WL/WM/WN`). For each period we pick the
lowest-cloud scene per tile and mosaic them together — a single "best scene" is not enough.

Bands:
- `red` (B04) + `nir` (B08) — for NDVI, kept as raw reflectance digital numbers (the
  NDVI ratio is scale-invariant, so no need to convert to physical reflectance)
- `visual` (true-colour TCI) — for display

Reads are done as **decimated windowed reads** (~100 m target resolution) directly against
the remote COGs, so only a small, already-downsampled amount of data is transferred per tile.

Output:
- `data/raw/Sentinel2/negros_s2.nc` — full mosaic, ~100 m, dims `(time, latitude, longitude)`
- `data/processed/day_2/negros_sentinel2.nc` — coarsened ~200 m, exercise-ready

In [ ]:
NEGROS_BBOX = [122.3, 9.8, 123.4, 11.0]   # central+full Negros, [W, S, E, N]
TARGET_RES_M = 100
MGRS_TILES = ["51PVL", "51PVM", "51PVN", "51PWL", "51PWM", "51PWN"]

catalog = pystac_client.Client.open("https://earth-search.aws.element84.com/v1")


def best_items_per_tile(date_range, cloud_max=60):
    """For each of the 6 MGRS tiles covering Negros, find the lowest-cloud scene in date_range."""
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=NEGROS_BBOX,
        datetime=date_range,
        query={"eo:cloud_cover": {"lt": cloud_max}},
        max_items=300,
    )
    items = list(search.items())
    tiles = {}
    for it in items:
        mgrs = it.id.split("_")[1]
        tiles.setdefault(mgrs, []).append(it)

    chosen = {}
    for t in MGRS_TILES:
        cands = tiles.get(t, [])
        if not cands:
            print(f"  WARNING: no scenes found for tile {t}")
            continue
        chosen[t] = min(cands, key=lambda x: x.properties.get("eo:cloud_cover", 99))
        it = chosen[t]
        print(f"  {t}: {it.id}  ({it.properties['datetime'][:10]}, cloud={it.properties.get('eo:cloud_cover', 0):.1f}%)")
    return chosen


print("Searching 2016-2018 (historical):")
items_old = best_items_per_tile("2016-01-01/2018-12-31")

print("\nSearching 2023-2024 (recent):")
items_new = best_items_per_tile("2023-01-01/2024-12-31")

In [ ]:
BANDS = ["red", "nir", "visual"]
FALLBACKS = {"nir": ["nir", "nir08", "B08"], "red": ["red", "B04"], "visual": ["visual", "tci"]}


def read_band_decimated(href, target_res_m=TARGET_RES_M):
    """Windowed + decimated read of a remote COG, cropped to NEGROS_BBOX and resampled
    directly to ~target_res_m during the read (GDAL uses COG overviews -> cheap over the network)."""
    with rasterio.open(href) as src:
        t = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        xmin, ymin = t.transform(NEGROS_BBOX[0], NEGROS_BBOX[1])
        xmax, ymax = t.transform(NEGROS_BBOX[2], NEGROS_BBOX[3])
        win = from_bounds(xmin, ymin, xmax, ymax, src.transform).intersection(
            rasterio.windows.Window(0, 0, src.width, src.height))

        decim = max(1, round(target_res_m / src.res[0]))
        out_h = max(1, int(win.height / decim))
        out_w = max(1, int(win.width / decim))
        data = src.read(window=win, out_shape=(src.count, out_h, out_w), resampling=Resampling.average)

        win_transform = src.window_transform(win)
        new_transform = win_transform * rasterio.Affine.scale(win.width / out_w, win.height / out_h)
        profile = src.profile.copy()
        profile.update(width=out_w, height=out_h, transform=new_transform, count=src.count, dtype=data.dtype)
    return data, profile


def download_period(label, chosen):
    """Download+mosaic red/nir/visual for one time period, reprojected to EPSG:4326."""
    out_dir = RAW_DIR / "Sentinel2" / label
    out_dir.mkdir(parents=True, exist_ok=True)

    mosaics = {}
    for band in BANDS:
        srcs = []
        for tile, item in chosen.items():
            out_tif = out_dir / f"{tile}_{band}.tif"
            if not out_tif.exists():
                asset_key = next((k for k in FALLBACKS.get(band, [band]) if k in item.assets), None)
                if asset_key is None:
                    print(f"  WARNING: {band!r} not found for {tile}; available: {list(item.assets.keys())}")
                    continue
                data, profile = read_band_decimated(item.assets[asset_key].href)
                profile.update(nodata=0)
                with rasterio.open(out_tif, "w", **profile) as dst:
                    dst.write(data)
                print(f"  saved {label}/{out_tif.name}  {data.shape}")
            srcs.append(rasterio.open(out_tif))

        mosaic, out_transform = rio_merge(srcs, res=(TARGET_RES_M, TARGET_RES_M), method="first")
        profile_out = srcs[0].profile.copy()
        profile_out.update(height=mosaic.shape[1], width=mosaic.shape[2],
                            transform=out_transform, count=mosaic.shape[0], nodata=0)
        for s in srcs:
            s.close()

        # wrap the mosaic (still in native UTM) and reproject to plain lat/lon (EPSG:4326)
        with MemoryFile() as memfile:
            with memfile.open(**profile_out) as mem:
                mem.write(mosaic)
            da = rioxarray.open_rasterio(memfile)
            da = da.astype("uint8" if band == "visual" else "float32")
            da = da.where(da != 0)  # mask nodata
            resamp = Resampling.nearest if band == "visual" else Resampling.bilinear
            da_ll = da.rio.reproject("EPSG:4326", resampling=resamp)

        mosaics[band] = da_ll
        print(f"  mosaicked {label}/{band}: {da_ll.shape}")

    return mosaics


mosaics_old = download_period("old", items_old)
mosaics_new = download_period("new", items_new)

In [ ]:
# Combine both periods into a single NetCDF with a 'time' dimension
def mosaics_to_dataset(mosaics, chosen):
    ds_vars = {}
    for band, da in mosaics.items():
        if band == "visual":
            for i, ch in enumerate(["visual_r", "visual_g", "visual_b"]):
                ds_vars[ch] = da.isel(band=i, drop=True)
        else:
            da2 = da.squeeze("band", drop=True) if "band" in da.dims and da.sizes.get("band", 1) == 1 else da
            ds_vars[band] = da2
    ds = xr.Dataset(ds_vars).rename({"x": "longitude", "y": "latitude"})
    rep_date = pd.Timestamp(min(it.properties["datetime"][:10] for it in chosen.values()))
    return ds.expand_dims(time=[rep_date])


ds_old = mosaics_to_dataset(mosaics_old, items_old)
ds_new = mosaics_to_dataset(mosaics_new, items_new)

combined = xr.concat([ds_old, ds_new], dim="time", join="outer")
combined.attrs["description"] = "Sentinel-2 L2A mosaic over Negros, red/nir/visual bands, ~100m"
combined.attrs["bands_note"] = ("red/nir are raw Sentinel-2 L2A surface reflectance digital numbers "
                                 "(NDVI=(nir-red)/(nir+red) works directly on these); "
                                 "visual_r/g/b are 0-255 true-colour RGB")
combined.attrs["tile_dates"] = str({
    "old": {t: it.properties["datetime"][:10] for t, it in items_old.items()},
    "new": {t: it.properties["datetime"][:10] for t, it in items_new.items()},
})

out_nc = RAW_DIR / "Sentinel2" / "negros_s2.nc"
combined.to_netcdf(out_nc)
print(f"Saved: {out_nc}")
print(combined)

In [ ]:
# --- Exercise-ready version: coarsen ~2x (100m -> ~200m) to keep the file small ---
day2_dir = PROCESSED_DIR / "day_2"
day2_dir.mkdir(parents=True, exist_ok=True)

ds_coarse = combined.coarsen(latitude=2, longitude=2, boundary="trim").mean()
for v in ds_coarse.data_vars:
    if v != "spatial_ref":
        ds_coarse[v] = ds_coarse[v].astype("float32")
ds_coarse.attrs = combined.attrs
ds_coarse.attrs["resolution_note"] = "coarsened ~2x from the ~100m native mosaic for a lighter workshop file (~200m)"

out_path = day2_dir / "negros_sentinel2.nc"
ds_coarse.to_netcdf(out_path)
print(f"Saved {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")

---
## 4. Hansen Global Forest Change — Negros

Hansen et al. (2013) provide annual forest cover and loss data from 2000 to present
at 30 m resolution as public COGs on Google Cloud Storage.

Layers:
- `treecover2000` — canopy cover in year 2000 (%)
- `lossyear` — year of first forest loss (1=2001 … 23=2023, 0=no loss)

Negros (9–11°N) straddles two 10°×10° tiles (`20N_120E` and `10N_120E`).
We use rasterio windowed reads on the remote COGs — only the Negros-sized patch
is transferred, not the full global tile.

In [ ]:
HANSEN_BASE   = 'https://storage.googleapis.com/earthenginepartners-hansen/GFC-2023-v1.11'
HANSEN_LAYERS = ['treecover2000', 'lossyear']
HANSEN_TILES  = ['20N_120E', '10N_120E']   # northern tile first (higher latitude)
NEGROS_FULL   = (121.5, 8.8, 124.5, 11.5)  # W, S, E, N — slightly wider than NEGROS_BBOX

out_dir = RAW_DIR / 'Hansen'
out_dir.mkdir(parents=True, exist_ok=True)

for layer in HANSEN_LAYERS:
    out_path = out_dir / f'hansen_{layer}_negros.tif'
    if out_path.exists():
        print(f'skip {out_path.name}')
        continue

    print(f'{layer}:')
    mem_files = []
    for tile in HANSEN_TILES:
        url = f'{HANSEN_BASE}/Hansen_GFC-2023-v1.11_{layer}_{tile}.tif'
        print(f'  reading {tile} ...', end='', flush=True)
        try:
            with rasterio.open(url) as src:
                win = from_bounds(*NEGROS_FULL, src.transform)
                row0 = max(0, int(win.row_off))
                col0 = max(0, int(win.col_off))
                row1 = min(src.height, int(win.row_off + win.height))
                col1 = min(src.width,  int(win.col_off + win.width))
                if row1 <= row0 or col1 <= col0:
                    print(' no overlap — skip')
                    continue
                win_c = rasterio.windows.Window(col0, row0, col1 - col0, row1 - row0)
                data = src.read(1, window=win_c)
                transform = src.window_transform(win_c)
                profile = src.profile.copy()
                profile.update(
                    width=data.shape[1], height=data.shape[0],
                    transform=transform, count=1,
                )
            mem = MemoryFile()
            with mem.open(**profile) as mds:
                mds.write(data, 1)
            mem_files.append(mem)
            print(f' {data.shape}')
        except Exception as e:
            print(f' ERROR: {e}')

    if not mem_files:
        print(f'  no data for {layer}')
        continue

    opened = [m.open() for m in mem_files]
    mosaic, out_transform = rio_merge(opened)
    profile_out = opened[0].profile.copy()
    profile_out.update(
        height=mosaic.shape[1], width=mosaic.shape[2],
        transform=out_transform, count=1,
    )
    for d in opened:
        d.close()
    for m in mem_files:
        m.close()

    with rasterio.open(out_path, 'w', **profile_out) as dst:
        dst.write(mosaic[0], 1)

    print(f'  -> saved {out_path.name}  {mosaic.shape}')

print('\nHansen download complete.')

In [ ]:
# --- Build exercise-ready netCDF: crop tight to NEGROS_BBOX, reproject to plain lat/lon, compact dtypes ---
day2_dir = PROCESSED_DIR / "day_2"
day2_dir.mkdir(parents=True, exist_ok=True)

layer_arrays = {}
for layer in HANSEN_LAYERS:
    da = rioxarray.open_rasterio(out_dir / f"hansen_{layer}_negros.tif").squeeze("band", drop=True)
    layer_arrays[layer] = da

hansen_ds = xr.Dataset(layer_arrays).rename({"x": "longitude", "y": "latitude"})
hansen_ds = hansen_ds.sel(longitude=slice(NEGROS_BBOX[0], NEGROS_BBOX[2]),
                           latitude=slice(NEGROS_BBOX[3], NEGROS_BBOX[1]))
hansen_ds["treecover2000"] = hansen_ds["treecover2000"].astype("uint8")
hansen_ds["lossyear"] = hansen_ds["lossyear"].astype("uint8")
hansen_ds.attrs["description"] = "Hansen Global Forest Change v1.11 (2023), cropped to Negros, native ~30m resolution"
hansen_ds.attrs["source"] = "Hansen et al. (2013), https://storage.googleapis.com/earthenginepartners-hansen/"
hansen_ds.attrs["lossyear_note"] = "year of first forest loss: 1=2001 ... 23=2023, 0=no loss detected"

out_nc = day2_dir / "negros_forest_change.nc"
hansen_ds.to_netcdf(out_nc)
print(f"Saved {out_nc}  ({out_nc.stat().st_size / 1e6:.1f} MB)")
print(hansen_ds)